# 03 · Build the `artists` table (Last.fm + MusicBrainz)

**Input:** `data/clean/events.csv` (from `02_events.ipynb`)
**Output:** `data/clean/artists.csv`, one row per `artist_key`

| Source | What we get | Speed |
|---|---|---|
| Last.fm `artist.getInfo` | listeners, playcount, genre tags, on tour, MusicBrainz ID | ~4 calls/sec |
| MusicBrainz | year formed, solo vs. group, country | **1 call/sec** (their rule) |

**Before running**, add to `.env`:
```
LASTFM_API_KEY=your_key
MB_CONTACT=your_email@virginia.edu   # MusicBrainz asks apps to include a contact
```

**Runtime:** about 15 min for Last.fm and about 1 hour for MusicBrainz with ~3,000 artists.
Every answer is saved to `data/interim/` as it arrives, so you can stop any time and re-run to pick up where it left off.
Set `LIMIT = 50` in the next cell to try it on a few artists first.

In [8]:
import json
import os
import re
import time
import unicodedata
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env", override=True)
LASTFM_KEY = os.getenv("LASTFM_API_KEY")
MB_CONTACT = os.getenv("MB_CONTACT", "student project")
assert LASTFM_KEY, "Add LASTFM_API_KEY to .env"

LIMIT = None             # e.g. 50 to test on a few artists first
SKIP_THEME_NIGHTS = True # tribute/party nights have no single artist to look up
CURRENT_YEAR = 2026

CLEAN, INTERIM = Path("data/clean"), Path("data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)
LFM_CACHE = INTERIM / "lastfm.jsonl"
MB_CACHE = INTERIM / "musicbrainz.jsonl"
print("Last.fm key found:", bool(LASTFM_KEY), "| MusicBrainz contact:", MB_CONTACT)

Last.fm key found: True | MusicBrainz contact: student project


## 1. List the artists to look up

In [9]:
events = pd.read_csv(CLEAN / "events.csv", low_memory=False)
ev = events[events["artist_key"].notna()]
if SKIP_THEME_NIGHTS:
    ev = ev[~ev["is_theme_night"]]

def most_common(s):
    return s.mode().iat[0] if not s.mode().empty else s.iat[0]

artists = (ev.groupby("artist_key")
             .agg(artist_name=("headliner_clean", most_common),
                  alt_name=("lookup_name_alt", most_common),
                  n_events=("event_id", "size"))
             .reset_index()
             .sort_values("n_events", ascending=False))
if LIMIT:
    artists = artists.head(LIMIT)
print(f"{len(artists)} artists to look up")
artists.head(10)

3372 artists to look up


,artist_key,artist_name,alt_name,n_events
3205,vince giordano and the nighthawks,Vince Giordano and the Nighthawks,Vince Giordano and the Nighthawks,94
328,birdland latin jazz orchestra,The Birdland Latin Jazz Orchestra,The Birdland Latin Jazz,49
240,backstage nashville daytime hit songwriters show,BACKSTAGE NASHVILLE! DAYTIME HIT SONGWRITERS SHOW,BACKSTAGE NASHVILLE! DAYTIME HIT SONGWRITERS SHOW,44
1325,high society new orleans jazz band,The High Society New Orleans Jazz Band,The High Society New Orleans Jazz,40
326,birdland big band,Birdland Big Band,Birdland,39
3040,time jumpers,THE TIME JUMPERS,THE TIME JUMPERS,33
2569,robert glasper,Robert Glasper,Robert Glasper,26
2333,pat barrett,Pat Barrett,Pat Barrett,25
1285,harlem gospel choir,Harlem Gospel Choir,Harlem Gospel Choir,24
757,david ostwald s louis armstrong eternity band,David Ostwald's Louis Armstrong Eternity Band,David Ostwald's Louis Armstrong Eternity,24


## 2. Helpers: cache, polite requests, name matching

In [10]:
def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {r["artist_key"]: r for r in rows}


def append_cache(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")


def get_json(url, params=None, headers=None, pause=0.25, tries=4):
    """GET with retries on rate limits and server hiccups. Returns (status_code, json or None)."""
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=20)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        time.sleep(pause)
        if r.status_code in (429, 500, 502, 503):
            time.sleep(3 * (attempt + 1))
            continue
        try:
            return r.status_code, r.json()
        except ValueError:
            return r.status_code, None
    return None, None


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9]+", " ", s.replace("&", " and ")).strip()
    return re.sub(r"^the\s+", "", s)


def similarity(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

## 3. Last.fm

Tries the artist name first, then the name without *Trio / Quartet / Big Band* if Last.fm doesn't know it. `autocorrect=1` fixes small spelling differences; the `name_similarity` column lets you catch wrong matches.

In [11]:
LFM_URL = "https://ws.audioscrobbler.com/2.0/"


def lastfm_lookup(name):
    status, js = get_json(LFM_URL, {"method": "artist.getinfo", "artist": name, "api_key": LASTFM_KEY,
                                    "autocorrect": 1, "format": "json"})
    if js is None:
        return "error", None
    if "error" in js:
        if js["error"] == 6:          # artist not found
            return "not_found", None
        if js["error"] in (10, 26):   # bad or suspended key: stop everything
            raise RuntimeError(f"Last.fm key problem: {js.get('message')}")
        return "error", js
    return "ok", js.get("artist")


cache = load_cache(LFM_CACHE)
todo = artists[~artists["artist_key"].isin(cache)]
print(f"Last.fm: {len(cache)} cached, {len(todo)} to fetch")

t0 = time.time()
for i, row in enumerate(todo.itertuples(), 1):
    status, data, used = "not_found", None, row.artist_name
    for name in dict.fromkeys([row.artist_name, row.alt_name]):   # unique, in order
        if not isinstance(name, str) or not name.strip():
            continue
        status, data = lastfm_lookup(name)
        used = name
        if status == "ok":
            break
    append_cache(LFM_CACHE, {"artist_key": row.artist_key, "query": used, "status": status, "data": data})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")
print("Last.fm done.")

Last.fm: 3372 cached, 0 to fetch
Last.fm done.


In [12]:
def parse_lastfm(r):
    a = r.get("data") or {}
    tags = (a.get("tags") or {}).get("tag") or []
    if isinstance(tags, dict):
        tags = [tags]
    tag_names = [t.get("name", "").lower() for t in tags]
    stats = a.get("stats") or {}
    return {
        "artist_key": r["artist_key"],
        "lastfm_status": r["status"],
        "lastfm_query": r["query"],
        "lastfm_name": a.get("name"),
        "mbid": a.get("mbid") or None,
        "listeners": pd.to_numeric(stats.get("listeners"), errors="coerce"),
        "playcount": pd.to_numeric(stats.get("playcount"), errors="coerce"),
        "on_tour": a.get("ontour") == "1",
        "tags": "; ".join(tag_names[:5]) or None,
        "top_tag": tag_names[0] if tag_names else None,
    }


lfm = pd.DataFrame(parse_lastfm(r) for r in load_cache(LFM_CACHE).values())
lfm["name_similarity"] = [similarity(q, n) if isinstance(n, str) else np.nan
                          for q, n in zip(lfm["lastfm_query"], lfm["lastfm_name"])]
print(lfm["lastfm_status"].value_counts().to_string())
print("Has an mbid:", lfm["mbid"].notna().mean().round(2))

lastfm_status
ok           2701
not_found     671
Has an mbid: 0.58


## 4. MusicBrainz
Looks up by `mbid` when Last.fm gave one; otherwise searches by name and only accepts a near-exact match (score 95+).

In [13]:
MB_BASE = "https://musicbrainz.org/ws/2/artist"
MB_HEADERS = {"User-Agent": f"DS6021TicketPriceProject/0.1 ( {MB_CONTACT} )", "Accept": "application/json"}


def mb_lookup(mbid, name):
    if mbid:
        status, js = get_json(f"{MB_BASE}/{mbid}", {"fmt": "json"}, MB_HEADERS, pause=1.1)
        if status == 200 and js:
            return "ok_mbid", js
    if isinstance(name, str) and name.strip():
        q = 'artist:"' + name.replace('"', "") + '"'
        status, js = get_json(MB_BASE, {"query": q, "fmt": "json", "limit": 3}, MB_HEADERS, pause=1.1)
        hits = (js or {}).get("artists") or []
        if hits and int(hits[0].get("score", 0)) >= 95:
            return "ok_search", hits[0]
        return "not_found", None
    return "not_found", None


lfm_by_key = lfm.set_index("artist_key")
mb_cache = load_cache(MB_CACHE)
todo = artists[~artists["artist_key"].isin(mb_cache)]
print(f"MusicBrainz: {len(mb_cache)} cached, {len(todo)} to fetch (~{len(todo) * 1.2 / 60:.0f} min)")

t0 = time.time()
for i, row in enumerate(todo.itertuples(), 1):
    info = lfm_by_key.loc[row.artist_key] if row.artist_key in lfm_by_key.index else None
    mbid = info["mbid"] if info is not None and isinstance(info["mbid"], str) else None
    name = info["lastfm_name"] if info is not None and isinstance(info["lastfm_name"], str) else row.artist_name
    status, data = mb_lookup(mbid, name)
    append_cache(MB_CACHE, {"artist_key": row.artist_key, "status": status, "data": data})
    if i % 100 == 0:
        print(f"  {i}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")
print("MusicBrainz done.")

MusicBrainz: 3372 cached, 0 to fetch (~0 min)
MusicBrainz done.


In [14]:
def parse_mb(r):
    a = r.get("data") or {}
    begin = (a.get("life-span") or {}).get("begin") or ""
    year = int(begin[:4]) if begin[:4].isdigit() else np.nan
    return {"artist_key": r["artist_key"], "mb_status": r["status"], "mb_id": a.get("id"),
            "mb_type": a.get("type"), "country": a.get("country"), "year_formed": year}


mb = pd.DataFrame(parse_mb(r) for r in load_cache(MB_CACHE).values())
print(mb["mb_status"].value_counts().to_string())

mb_status
ok_mbid      1941
not_found     985
ok_search     446


## 5. Combine and save

In [15]:
out = (artists.merge(lfm, on="artist_key", how="left")
               .merge(mb, on="artist_key", how="left"))

out["log_listeners"] = np.log1p(out["listeners"])
out["plays_per_listener"] = out["playcount"] / out["listeners"].replace(0, np.nan)
out["years_active"] = (CURRENT_YEAR - out["year_formed"]).clip(lower=0)
out["artist_type"] = out["mb_type"].map({"Person": "solo", "Group": "group", "Orchestra": "group",
                                         "Choir": "group"}).fillna("unknown")
out["mbid"] = out["mbid"].fillna(out["mb_id"])
out["low_confidence_match"] = out["name_similarity"] < 0.6

cols = ["artist_key", "artist_name", "n_events", "lastfm_status", "lastfm_name", "name_similarity",
        "low_confidence_match", "listeners", "log_listeners", "playcount", "plays_per_listener",
        "on_tour", "tags", "top_tag", "mbid", "mb_status", "artist_type", "country",
        "year_formed", "years_active"]
out = out[cols]
out.to_csv(CLEAN / "artists.csv", index=False)

print(f"Saved {CLEAN / 'artists.csv'}: {len(out)} artists")
print(f"  Last.fm found:      {out['lastfm_status'].eq('ok').mean():.0%}")
print(f"  MusicBrainz found:  {out['mb_status'].str.startswith('ok').mean():.0%}")
print(f"  has year_formed:    {out['year_formed'].notna().mean():.0%}")
print(f"  low-confidence:     {out['low_confidence_match'].sum()} (check below)")
events_covered = events["artist_key"].isin(out.loc[out["lastfm_status"].eq("ok"), "artist_key"]).mean()
print(f"  share of events with Last.fm data: {events_covered:.0%}")

Saved data\clean\artists.csv: 3372 artists
  Last.fm found:      80%
  MusicBrainz found:  71%
  has year_formed:    38%
  low-confidence:     3 (check below)
  share of events with Last.fm data: 75%


In [16]:
# Possible wrong matches: query vs. what Last.fm returned. Fix by editing or dropping these rows.
out.loc[out["low_confidence_match"], ["artist_name", "lastfm_name", "name_similarity", "listeners"]].head(25)

,artist_name,lastfm_name,name_similarity,listeners
50,80's,[unknown],0.00,1127865.0
176,The Rocky Horror Picture Show,Richard O'Brien,0.25,281771.0
1301,Eivor,Eivør Pálsdóttir,0.50,30328.0


In [17]:
# Sanity check: most popular artists in the data
out.nlargest(15, "listeners")[["artist_name", "listeners", "top_tag", "artist_type", "years_active", "n_events"]]

,artist_name,listeners,top_tag,artist_type,years_active,n_events
268,SABBATH,4524119.0,heavy metal,group,58.0,3
936,Tyga,2812290.0,hip-hop,solo,37.0,2
3027,The Temptations,2434100.0,soul,group,66.0,1
1394,Icona Pop,2297180.0,electronic,group,17.0,1
1481,Drowning Pool,2229612.0,nu metal,group,30.0,1
3022,Thundercat,2157157.0,funk,solo,42.0,1
1580,DJ Shadow,2118547.0,trip-hop,solo,54.0,1
1026,Fitz and The Tantrums,2001656.0,soul,group,18.0,2
3287,Wheatus,1991570.0,alternative rock,group,31.0,1
3034,Thievery Corporation,1940448.0,chillout,group,31.0,1
